In [1]:
import os

os.environ["WANDB_DISABLED"] = "true"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["USE_TF"] = "0"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

In [2]:
import pandas as pd
import glob
import os
from huggingface_hub import HfFileSystem

# ==========================================
# 1. LOAD LOCAL METADATA (MEMORY-SAFE MODE)
# ==========================================
print("Loading local Metadata (Bypassing RAM limits)...")
folder_path = "./Amazon_review_ds_elec/"
file_list = glob.glob(os.path.join(folder_path, "*.parquet"))

# ONLY load these 4 columns to prevent OSError 12
needed_columns = ["parent_asin", "title", "store", "main_category"]
final_df_list = [pd.read_parquet(file, columns=needed_columns) for file in file_list]
full_final_df = pd.concat(final_df_list, ignore_index=True)
print(f"Loaded {len(full_final_df):,} metadata rows safely.")

# ==========================================
# 2. DYNAMICALLY FIND & STREAM ONE REVIEWS CHUNK
# ==========================================
# print("\nSearching Hugging Face for the exact Review Parquet paths...")
# fs = HfFileSystem()
#
# # The ** wildcard bypasses the folder guessing game entirely
# review_files = fs.glob("datasets/McAuley-Lab/Amazon-Reviews-2023/**/raw_review_Electronics/*.parquet")
# first_file = "hf://" + review_files[0]
# print(f"Found the reviews! Streaming into pandas: {first_file}")

# read_parquet natively streams from hf:// paths without full disk downloads
# reviews_chunk = pd.read_parquet(first_file)

# Take a 100k sample to keep the merge fast
# reviews_sample = reviews_chunk.sample(n=100000, random_state=42)
# print(f"Loaded {len(reviews_sample):,} reviews successfully.")
#
# # ==========================================
# # 3. MERGE THE GRAPH EDGES
# # ==========================================
# merged_final_df = pd.merge(
#     reviews_sample,
#     full_final_df,
#     on='parent_asin',
#     how='inner'
# )
#
# final_final_df = merged_final_df.rename(columns={
#     'store': 'brand',
#     'main_category': 'category',
#     'text': 'reviews_text',
#     'title': 'name'
# })

# print(f"\nFinal dataset ready! Total matched rows: {len(final_final_df):,}")
# display(final_final_df.head(3))

Loading local Metadata (Bypassing RAM limits)...
Loaded 1,610,012 metadata rows safely.


In [3]:
import pandas as pd

# 1. Point this to the 6GB file you just downloaded
reviews_file_path = "./Electronics.jsonl.gz"

print("Loading a 100k sample from the local 6GB reviews file...")

# 2. Use chunksize to read just the first 100,000 rows without exploding your RAM
json_reader = pd.read_json(reviews_file_path, lines=True, chunksize=100000)
reviews_sample = next(json_reader)

print(f"Successfully loaded {len(reviews_sample):,} reviews into memory!")

# 3. Merge with your local metadata (full_final_df) that you already loaded successfully
merged_final_df = pd.merge(reviews_sample, full_final_df, on="parent_asin", how="inner")

# 4. Rename columns to match your existing Keras pipeline inputs
final_df = merged_final_df.rename(
    columns={
        "store": "brand",
        "main_category": "category",
        "text": "reviews_text",
        "title": "name",
    }
)


print(f"\nFinal dataset ready! Total matched rows: {len(final_df):,}")
display(final_df.head(3))

Loading a 100k sample from the local 6GB reviews file...
Successfully loaded 100,000 reviews into memory!

Final dataset ready! Total matched rows: 100,000


,rating,title_x,reviews_text,images,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase,title_y,brand,category
0,3,Smells like gasoline! Going back!,First & most offensive: they reek of gasoline ...,[{'small_image_url': 'https://m.media-amazon.c...,B083NRGZMM,B083NRGZMM,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,2022-07-18 22:58:37.948,0,True,"Binoculars, 12x42 Binoculars for Adults, Binoc...",Hikkogo,Camera & Photo
1,1,Didn’t work at all lenses loose/broken.,These didn’t work. Idk if they were damaged in...,[],B07N69T6TM,B07N69T6TM,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,2020-06-20 18:42:29.731,0,True,"Toys for 4-5 Year Old Boys, Mom&myaboys 8 X 21...",mom&myaboys,Camera & Photo
2,5,Excellent!,I love these. They even come with a carry case...,[],B01G8JO5F2,B01G8JO5F2,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,2018-04-07 09:23:37.534,0,True,"Senso Bluetooth Headphones, Best Wireless Spor...",Senso,All Electronics


In [4]:
# 1. Count how many ratings each item (parent_asin) has received
item_counts = final_df["parent_asin"].value_counts()

# 2. Define the threshold: items with fewer than 5 ratings are "Cold"
cold_items = item_counts[item_counts < 5].index.tolist()
head_items = item_counts[item_counts >= 5].index.tolist()

print(f"Total Unique Items: {len(item_counts):,}")
print(f"Head Items (5+ ratings): {len(head_items):,}")
print(f"Cold Items (<5 ratings): {len(cold_items):,}")

# 3. Create explicit evaluation datasets based on the split
final_df_cold = final_df[final_df["parent_asin"].isin(cold_items)].copy()
final_df_head = final_df[final_df["parent_asin"].isin(head_items)].copy()


final_df_cold = final_df_cold.rename(columns={"title_y": "name"})
final_df_head = final_df_head.rename(columns={"title_y": "name"})
print(f"\nYour 'Cold-Start' evaluation set contains {len(final_df_cold):,} rows.")

Total Unique Items: 59,429
Head Items (5+ ratings): 2,434
Cold Items (<5 ratings): 56,995

Your 'Cold-Start' evaluation set contains 74,000 rows.


In [5]:
final_df["user_id_encoded"] = final_df["user_id"].astype("category").cat.codes
final_df["item_id_encoded"] = final_df["parent_asin"].astype("category").cat.codes

# Rename the product title from the metadata to 'name'
final_df = final_df.rename(columns={"title_y": "name"})

# Verify it worked
# print(final_df[['name', 'reviews_text', 'brand_encoded', 'category_encoded']].head(2))

In [ ]:
import tensorflow as tf
from transformers import TFBertModel, BertTokenizer

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

In [ ]:
def build_joint_explainability_model(
    num_entities, num_relations, vocab_size, max_seq_len=128, embed_dim=64
):
    input_ids = tf.keras.Input(shape=(max_seq_len,), dtype="int32", name="input_ids")
    attn_mask = tf.keras.Input(
        shape=(max_seq_len,), dtype="int32", name="attention_mask"
    )

    bert = TFBertModel.from_pretrained("bert-base-uncased")
    bert.trainable = False
    bert_output = bert(input_ids=input_ids, attention_mask=attn_mask)
    bert_pooled = bert_output.pooler_output

    bert_compressed = tf.keras.layers.Dense(
        64, activation="relu", name="bert_compression"
    )(bert_pooled)
    head_idx = tf.keras.layers.Input(shape=(1,), dtype="int32", name="head_idx")
    # rel_idx = tf.keras.layers.Input(shape=(1,), dtype="int32", name = 'rel_idx')
    tail_idx = tf.keras.layers.Input(shape=(1,), dtype="int32", name="tail_idx")
    brand_idx = tf.keras.layers.Input(shape=(1,), dtype="int32", name="brand_idx")
    cat_idx = tf.keras.layers.Input(shape=(1,), dtype="int32", name="cat_idx")

    entity_emb_layer = tf.keras.layers.Embedding(
        input_dim=num_entities, output_dim=embed_dim, name="entity_emb"
    )
    # relation_emb_layer = tf.keras.layers.Embedding(input_dim = num_relations, output_dim = embed_dim, name = 'relation_emb')

    h_emb = tf.keras.layers.Flatten()(entity_emb_layer(head_idx))
    # r_emb = tf.keras.layers.Flatten()(relation_emb_layer(rel_idx))
    t_emb = tf.keras.layers.Flatten()(entity_emb_layer(tail_idx))
    b_emb = tf.keras.layers.Flatten()(entity_emb_layer(brand_idx))
    c_emb = tf.keras.layers.Flatten()(entity_emb_layer(cat_idx))

    merged = tf.keras.layers.concatenate([h_emb, t_emb, b_emb, c_emb, bert_compressed])

    fusion_dense = tf.keras.layers.Dense(256, activation="relu", name="fusion_dense")(
        merged
    )
    fusion_state = tf.keras.layers.Dropout(0.2)(fusion_dense)

    rating_output = tf.keras.layers.Dense(1, activation="linear", name="rating_output")(
        fusion_state
    )

    decoder_inputs = tf.keras.layers.Input(
        shape=(None,), dtype="int32", name="decoder_inputs"
    )

    lstm_initial_state = tf.keras.layers.Dense(
        512, activation="tanh", name="lstm_initial_state"
    )(fusion_state)

    decoder_embedding = tf.keras.layers.Embedding(
        input_dim=vocab_size, output_dim=256, name="decoder_embedding"
    )(decoder_inputs)

    decoder_lstm = tf.keras.layers.LSTM(
        512, return_sequences=True, activation="tanh", name="justification_lstm"
    )
    lstm_out = decoder_lstm(
        decoder_embedding, initial_state=[lstm_initial_state, lstm_initial_state]
    )

    justification_output = tf.keras.layers.Dense(
        vocab_size, name="justification_output"
    )(lstm_out)

    model = tf.keras.Model(
        inputs=[
            head_idx,  # User ID
            tail_idx,  # Item ID
            brand_idx,  # Brand ID
            cat_idx,  # Category ID
            input_ids,  # BERT token IDs (Encoder)
            attn_mask,  # BERT padding mask (Encoder)
            decoder_inputs,  # LSTM shifted targets (Decoder)
        ],
        outputs=[rating_output, justification_output],
        name="Explainable_Hybrid_Recommender",
    )

    return model

In [ ]:
def build_baseline_ablation_model(
    num_entities, vocab_size, max_seq_len=128, embed_dim=64
):
    # 1. Text Encoder Inputs
    input_ids = tf.keras.Input(shape=(max_seq_len,), dtype="int32", name="input_ids")
    attn_mask = tf.keras.Input(
        shape=(max_seq_len,), dtype="int32", name="attention_mask"
    )

    # BERT Encoding
    bert = TFBertModel.from_pretrained("bert-base-uncased")
    bert.trainable = False
    bert_output = bert(input_ids=input_ids, attention_mask=attn_mask)
    bert_compressed = tf.keras.layers.Dense(
        64, activation="relu", name="bert_compression"
    )(bert_output.pooler_output)

    # 2. Collaborative Filtering Inputs (NO KNOWLEDGE GRAPH)
    head_idx = tf.keras.Input(shape=(1,), dtype="int32", name="head_idx")
    tail_idx = tf.keras.Input(shape=(1,), dtype="int32", name="tail_idx")

    entity_emb_layer = tf.keras.layers.Embedding(
        input_dim=num_entities, output_dim=embed_dim, name="entity_emb"
    )

    h_emb = tf.keras.layers.Flatten()(entity_emb_layer(head_idx))
    t_emb = tf.keras.layers.Flatten()(entity_emb_layer(tail_idx))

    # 3. Fusion WITHOUT Brand or Category
    merged = tf.keras.layers.concatenate([h_emb, t_emb, bert_compressed])
    fusion_dense = tf.keras.layers.Dense(256, activation="relu", name="fusion_dense")(
        merged
    )
    fusion_state = tf.keras.layers.Dropout(0.2)(fusion_dense)

    # 4. Rating Head
    rating_output = tf.keras.layers.Dense(1, activation="linear", name="rating_output")(
        fusion_state
    )

    # 5. NLG Head
    decoder_inputs = tf.keras.Input(shape=(None,), dtype="int32", name="decoder_inputs")
    lstm_initial_state = tf.keras.layers.Dense(
        512, activation="tanh", name="lstm_initial_state"
    )(fusion_state)
    decoder_embedding = tf.keras.layers.Embedding(
        input_dim=vocab_size, output_dim=256, name="decoder_embedding"
    )(decoder_inputs)

    decoder_lstm = tf.keras.layers.LSTM(
        512, return_sequences=True, activation="tanh", name="justification_lstm"
    )
    lstm_out = decoder_lstm(
        decoder_embedding, initial_state=[lstm_initial_state, lstm_initial_state]
    )

    justification_output = tf.keras.layers.Dense(
        vocab_size, name="justification_output"
    )(lstm_out)

    # Compile the Baseline
    baseline_model = tf.keras.Model(
        inputs=[head_idx, tail_idx, input_ids, attn_mask, decoder_inputs],
        outputs=[rating_output, justification_output],
        name="Baseline_No_KG_Recommender",
    )

    return baseline_model


baseline_model = build_joint_explainability_model(
    num_entities, num_relations, vocab_size, 128, 64
)
baseline_model.summary()

baseline_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=2e-5, clipnorm=1.0),
    loss={
        "rating_output": "mse",
        "justification_output": tf.keras.losses.SparseCategoricalCrossentropy(
            from_logits=True
        ),
    },
    loss_weights={"rating_output": 1, "justification_output": 0.5},
    metrics={"rating_output": "mae"},
)

In [ ]:
vocab_size = tokenizer.vocab_size

# 1. Cast initial max IDs to strict integers
max_user_id = int(final_df["user_id_encoded"].max())
max_item_id = int(final_df["item_id_encoded"].max())

print("maxuserid ", max_user_id)
print("maxitemid ", max_item_id)
num_entities = max(max_user_id, max_item_id)

# 2. Handle Brands (Fill NaNs with 0 and cast to int)
brands_unique = final_df["brand"].dropna().unique()
brands_map = {brand: idx + num_entities + 1 for idx, brand in enumerate(brands_unique)}

final_df["brand_encoded"] = final_df["brand"].map(brands_map).fillna(0).astype(int)
print(final_df["brand_encoded"].head(3))

num_entities = max(num_entities, int(final_df["brand_encoded"].max()))

# 3. Handle Categories (Fix column name, fill NaNs, cast to int)
cats_unique = final_df["category"].dropna().unique()
cats_map = {
    category: idx + num_entities + 1 for idx, category in enumerate(cats_unique)
}

final_df["category_encoded"] = final_df["category"].map(cats_map).fillna(0).astype(int)

# 4. Final strict integer cast for Keras
num_entities = int(max(num_entities, final_df["category_encoded"].max())) + 1
num_relations = 3

print(
    f"\nFinal num_entities passed to Keras: {num_entities} (Type: {type(num_entities)})"
)

# Build and compile the model

In [ ]:
import numpy as np
from transformers import BertTokenizer


def create_tf_dataset(
    final_df, tokenizer, max_len=128, batch_size=16, is_training=True
):
    encoder_tokenized = tokenizer(
        final_df["name"].tolist(),
        truncation=True,
        padding="max_length",
        max_length=max_len,
        return_tensors="np",
    )

    print(f"Tokenizinf {len(final_df)}  Reviews")

    decoder_tokenized = tokenizer(
        final_df["reviews_text"].tolist(),
        truncation=True,
        padding="max_length",
        max_length=max_len,
        return_tensors="np",
    )

    input_ids = encoder_tokenized["input_ids"]
    attn_mask = encoder_tokenized["attention_mask"]

    decoder_inputs = decoder_tokenized["input_ids"][:, :-1]
    justification_targets = decoder_tokenized["input_ids"][:, 1:]

    inputs = {
        "input_ids": input_ids,
        "attention_mask": attn_mask,
        "decoder_inputs": decoder_inputs,
        "head_idx": final_df["user_id_encoded"].values.reshape(-1, 1),
        # "rel_idx": final_df['relation_encoded'].values.reshape(-1, 1),
        "tail_idx": final_df["item_id_encoded"].values.reshape(-1, 1),
        "brand_idx": final_df["brand_encoded"].values.reshape(-1, 1),
        "cat_idx": final_df["category_encoded"].values.reshape(-1, 1),
    }

    targets = {
        "rating_output": final_df["rating"].values.reshape(-1, 1),
        "justification_output": justification_targets,
    }
    justification_weights = (justification_targets != 0).astype(np.float32)
    rating_weight = np.ones((len(final_df), 1), dtype=np.float32)
    sample_weights = {
        "rating_output": rating_weight,
        "justification_output": justification_weights,
    }
    dataset = tf.data.Dataset.from_tensor_slices((inputs, targets, sample_weights))
    if is_training:
        dataset = dataset.shuffle(buffer_size=len(final_df), seed=42)
    dataset = dataset.batch(batch_size)
    dataset = dataset.prefetch(tf.data.AUTOTUNE)
    return dataset

In [ ]:
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
final_df_shuffled = final_df.sample(frac=1, random_state=42).reset_index(drop=True)
split_idx = int(len(final_df_shuffled) * 0.8)


train_idx = final_df_shuffled.iloc[:split_idx]
val_idx = final_df_shuffled.iloc[split_idx:]

train_dataset = create_tf_dataset(
    train_idx, tokenizer, max_len=128, batch_size=16, is_training=True
)

val_dataset = create_tf_dataset(
    val_idx, tokenizer, max_len=128, batch_size=16, is_training=False
)

# history = model.fit(
#     train_dataset,
#     validation_data=val_dataset,
#     epochs=15,
#     callbacks = [tf.keras.callbacks.EarlyStopping(monitor = 'val_loss', patience = 3, restore_best_weights=True)],
# )
# #

In [ ]:
import nltk
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction


def evaluate_nlg_performance(model, val_dataset, tokenizer):
    print("Evaluating NLG")

    bleu_scores = []
    smoother = SmoothingFunction().method1
    for batch_inputs, batch_targets, batch_weights in val_dataset:
        predictions = model(batch_inputs, training=False)
        justification_logits = predictions[1]
        pred_token_ids = tf.argmax(justification_logits, axis=-1).numpy()
        true_token_ids = batch_targets["justification_output"].numpy()

        for i in range(0, len(true_token_ids)):
            true_text = tokenizer.decode(
                true_token_ids[i], skip_special_tokens=True
            ).split()
            pred_text = tokenizer.decode(
                pred_token_ids[i], skip_special_tokens=True
            ).split()
            if len(true_text) > 0:
                score = sentence_bleu(
                    [true_text], pred_text, smoothing_function=smoother
                )
                bleu_scores.append(score)
    final_score = np.mean(bleu_scores)
    print("Final Score = ", final_score)
    print("\n--- Example Generation ---")
    print(f"Target: {' '.join(true_text)}")
    print(f"Output: {' '.join(pred_text)}")

    return final_score


# val_bleu = evaluate_nlg_performance(model, val_dataset, tokenizer)
# print(f'BLEU Score = {val_bleu}')

In [ ]:
model.save_weights("actual_hybrid_weights.h5")
baseline_model.save_weights("actual_hybrid_baseline_weights.h5")
transformer_model.save_weights("actual_hybrid_transformer_weights.h5")

In [ ]:
def generate_justification(
    model,
    user_id,
    item_id,
    brand_id,
    cat_id,
    item_name,
    tokenizer,
    max_length=30,
    temperature=0.7,
    top_k=50,
):
    generated_sequence = [tokenizer.cls_token_id]

    head_tensor = tf.constant([[user_id]], dtype=tf.int32)
    tail_tensor = tf.constant([[item_id]], dtype=tf.int32)
    b_tensor = tf.constant([[brand_id]], dtype=tf.int32)
    c_tensor = tf.constant([[cat_id]], dtype=tf.int32)

    # Encode the actual product name to feed to BERT
    name_tokens = tokenizer(
        item_name,
        truncation=True,
        padding="max_length",
        max_length=128,
        return_tensors="tf",
    )
    input_ids_tensor = name_tokens["input_ids"]
    attn_mask_tensor = name_tokens["attention_mask"]

    for _ in range(max_length):
        decoder_input = tf.constant([generated_sequence], dtype=tf.int32)

        predictions = model(
            [
                head_tensor,
                tail_tensor,
                b_tensor,
                c_tensor,
                input_ids_tensor,
                attn_mask_tensor,
                decoder_input,
            ],
            training=False,
        )

        next_token_logits = predictions[1][0, -1, :] / temperature
        top_k_logits, top_k_indices = tf.math.top_k(next_token_logits, k=top_k)
        gumbel_probs = tf.nn.softmax(top_k_logits).numpy()
        chosen_idx = np.random.choice(len(top_k_indices), p=gumbel_probs)
        next_token_id = top_k_indices[chosen_idx].numpy()

        if next_token_id == tokenizer.sep_token_id:
            break

        generated_sequence.append(next_token_id)

    return tokenizer.decode(generated_sequence, skip_special_tokens=True)

In [ ]:
# Grab a sample from your validation split
sample = val_idx.iloc[0]

gen_text = generate_justification(
    transformer_model,
    user_id=sample["user_id_encoded"],
    item_id=sample["item_id_encoded"],
    brand_id=sample["brand_encoded"],
    cat_id=sample["category_encoded"],
    item_name=sample["name"],  # <-- Added this line!
    tokenizer=tokenizer,
    max_length=12,
    temperature=0.1,  # <-- Drop from 0.7 to 0.2
    top_k=5,
)

print("🎯 TRUE REVIEW:")
print(sample["reviews_text"])
print("\n🤖 GENERATED JUSTIFICATION:")
print(gen_text)

In [ ]:
print("Starting Baseline Ablation Training (No KG Features)...")

baseline_history = baseline_model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=15,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss", patience=3, restore_best_weights=True
        )
    ],
)

In [ ]:
def generate_baseline_justification(
    model,
    user_id,
    item_id,
    item_name,
    tokenizer,
    max_length=30,
    temperature=0.7,
    top_k=50,
):
    generated_sequence = [tokenizer.cls_token_id]
    head_tensor = tf.constant([[user_id]], dtype=tf.int32)
    tail_tensor = tf.constant([[item_id]], dtype=tf.int32)

    name_tokens = tokenizer(
        item_name,
        truncation=True,
        padding="max_length",
        max_length=128,
        return_tensors="tf",
    )
    input_ids_tensor = name_tokens["input_ids"]
    attn_mask_tensor = name_tokens["attention_mask"]

    for _ in range(max_length):
        decoder_input = tf.constant([generated_sequence], dtype=tf.int32)
        # Note: Brand and Category tensors removed to match Baseline architecture
        predictions = model(
            [
                head_tensor,
                tail_tensor,
                input_ids_tensor,
                attn_mask_tensor,
                decoder_input,
            ],
            training=False,
        )

        next_token_logits = predictions[1][0, -1, :] / temperature
        top_k_logits, top_k_indices = tf.math.top_k(next_token_logits, k=top_k)
        gumbel_probs = tf.nn.softmax(top_k_logits).numpy()
        chosen_idx = np.random.choice(len(top_k_indices), p=gumbel_probs)
        next_token_id = top_k_indices[chosen_idx].numpy()

        if next_token_id == tokenizer.sep_token_id:
            break
        generated_sequence.append(next_token_id)

    return tokenizer.decode(generated_sequence, skip_special_tokens=True)

In [ ]:
# 1. Re-split the evaluation datasets from the FULLY ENCODED final_df
item_counts = final_df["parent_asin"].value_counts()
cold_items = item_counts[item_counts < 5].index.tolist()
head_items = item_counts[item_counts >= 5].index.tolist()

final_df_cold = final_df[final_df["parent_asin"].isin(cold_items)].copy()
final_df_head = final_df[final_df["parent_asin"].isin(head_items)].copy()

# 2. Convert the updated DataFrames into TF Datasets
print("Preparing evaluation datasets...")
cold_dataset = create_tf_dataset(
    final_df_cold, tokenizer, max_len=128, batch_size=16, is_training=False
)
head_dataset = create_tf_dataset(
    final_df_head, tokenizer, max_len=128, batch_size=16, is_training=False
)

# 3. Evaluate Rating Accuracy (MAE)
print("\n--- RATING PREDICTION EVALUATION ---")
print("Evaluating on HEAD items (5+ ratings)...")
head_metrics = model.evaluate(head_dataset)

print("\nEvaluating on COLD items (<5 ratings)...")
cold_metrics = model.evaluate(cold_dataset)

# 4. Evaluate Justification Quality (BLEU)
print("\n--- TEXT GENERATION (NLG) EVALUATION ---")
print("Calculating BLEU score for HEAD items...")
head_bleu = evaluate_nlg_performance(model, head_dataset, tokenizer)

print("\nCalculating BLEU score for COLD items...")
cold_bleu = evaluate_nlg_performance(model, cold_dataset, tokenizer)

In [ ]:
import tensorflow as tf
from transformers import TFBertModel

# 1. Enable Mixed Precision to save RTX 5050 VRAM
tf.keras.mixed_precision.set_global_policy("mixed_float16")


# 2. Custom Wrapper for BERT
class BertWrapper(tf.keras.layers.Layer):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        self.bert = TFBertModel.from_pretrained("bert-base-uncased")
        self.bert.trainable = False

    def call(self, inputs):
        input_ids, attn_mask = inputs
        return self.bert(input_ids=input_ids, attention_mask=attn_mask).pooler_output


# 3. Custom Wrapper for Positional Embeddings
class TokenAndPositionEmbedding(tf.keras.layers.Layer):
    def __init__(self, vocab_size, max_seq_len, embed_dim, **kwargs):
        super().__init__(**kwargs)
        self.token_emb = tf.keras.layers.Embedding(
            input_dim=vocab_size, output_dim=embed_dim
        )
        self.pos_emb = tf.keras.layers.Embedding(
            input_dim=max_seq_len, output_dim=embed_dim
        )

    def call(self, x):
        maxlen = tf.shape(x)[-1]
        positions = tf.range(start=0, limit=maxlen, delta=1)
        return self.token_emb(x) + self.pos_emb(positions)


def build_transformer_explainability_model(
    num_entities, vocab_size, max_seq_len=128, embed_dim=64
):
    # --- ENCODER & FUSION ---
    input_ids = tf.keras.Input(shape=(max_seq_len,), dtype="int32", name="input_ids")
    attn_mask = tf.keras.Input(
        shape=(max_seq_len,), dtype="int32", name="attention_mask"
    )

    bert_pooled = BertWrapper(name="bert_wrapper")([input_ids, attn_mask])
    bert_compressed = tf.keras.layers.Dense(
        64, activation="relu", name="bert_compression"
    )(bert_pooled)

    head_idx = tf.keras.Input(shape=(1,), dtype="int32", name="head_idx")
    tail_idx = tf.keras.Input(shape=(1,), dtype="int32", name="tail_idx")
    brand_idx = tf.keras.Input(shape=(1,), dtype="int32", name="brand_idx")
    cat_idx = tf.keras.Input(shape=(1,), dtype="int32", name="cat_idx")

    entity_emb_layer = tf.keras.layers.Embedding(
        input_dim=num_entities, output_dim=embed_dim, name="entity_emb"
    )

    h_emb = tf.keras.layers.Flatten()(entity_emb_layer(head_idx))
    t_emb = tf.keras.layers.Flatten()(entity_emb_layer(tail_idx))
    b_emb = tf.keras.layers.Flatten()(entity_emb_layer(brand_idx))
    c_emb = tf.keras.layers.Flatten()(entity_emb_layer(cat_idx))

    merged = tf.keras.layers.concatenate([h_emb, t_emb, b_emb, c_emb, bert_compressed])
    fusion_dense = tf.keras.layers.Dense(256, activation="relu", name="fusion_dense")(
        merged
    )
    fusion_state = tf.keras.layers.Dropout(0.2)(fusion_dense)

    rating_output = tf.keras.layers.Dense(
        1, activation="linear", name="rating_output", dtype="float32"
    )(fusion_state)

    # --- THE STATE-OF-THE-ART TRANSFORMER DECODER ---
    decoder_inputs = tf.keras.Input(shape=(None,), dtype="int32", name="decoder_inputs")

    x = TokenAndPositionEmbedding(vocab_size, max_seq_len, 256, name="pos_embedding")(
        decoder_inputs
    )

    # 1. Masked Self-Attention
    attn1 = tf.keras.layers.MultiHeadAttention(num_heads=8, key_dim=64)(
        query=x, value=x, key=x, use_causal_mask=True
    )
    x = tf.keras.layers.LayerNormalization(dtype="float32")(x + attn1)
    x = tf.cast(x, "float16")  # Drop back to float16 to prevent type collisions

    # 2. Cross-Attention

    rating_16 = tf.cast(rating_output, "float16")
    sentiment_memory = tf.keras.layers.concatenate([fusion_state, rating_16])

    memory = tf.keras.layers.Reshape((1, 257))(sentiment_memory)

    attn2 = tf.keras.layers.MultiHeadAttention(num_heads=8, key_dim=64)(
        query=x, value=memory, key=memory
    )
    x = tf.keras.layers.LayerNormalization(dtype="float32")(x + attn2)
    x = tf.cast(x, "float16")  # Drop back to float16

    # 3. Feed Forward Network
    ffn = tf.keras.layers.Dense(512, activation="relu")(x)
    ffn = tf.keras.layers.Dense(256)(ffn)

    x = tf.keras.layers.LayerNormalization(dtype="float32")(x + ffn)
    # Leave the final x as float32 for softmax stability

    justification_output = tf.keras.layers.Dense(
        vocab_size, name="justification_output", dtype="float32"
    )(x)

    model = tf.keras.Model(
        inputs=[
            head_idx,
            tail_idx,
            brand_idx,
            cat_idx,
            input_ids,
            attn_mask,
            decoder_inputs,
        ],
        outputs=[rating_output, justification_output],
        name="Transformer_Hybrid_Recommender",
    )
    return model


transformer_model = build_transformer_explainability_model(
    num_entities, vocab_size, 128, 64
)

In [ ]:
print("\n--- BASELINE ABLATION EVALUATION (NO KG) ---")
print("Evaluating on HEAD items (5+ ratings)...")
base_head_metrics = baseline_model.evaluate(head_dataset)

print("\nEvaluating on COLD items (<5 ratings)...")
base_cold_metrics = baseline_model.evaluate(cold_dataset)

In [ ]:
transformer_model = build_transformer_explainability_model(
    num_entities, vocab_size, 128, 64
)
transformer_model.summary()

In [ ]:
# 1. Compile the Transformer model
transformer_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=2e-5, clipnorm=1.0),
    loss={
        "rating_output": "mse",
        "justification_output": tf.keras.losses.SparseCategoricalCrossentropy(
            from_logits=True
        ),
    },
    loss_weights={"rating_output": 1.0, "justification_output": 1.0},
    metrics={"rating_output": "mae"},
)

# 2. Dry run with batch size 8
test_dataset = create_tf_dataset(
    train_idx.iloc[:16], tokenizer, max_len=128, batch_size=8, is_training=False
)

for batch_inputs, batch_targets, _ in test_dataset.take(1):
    preds = transformer_model(batch_inputs, training=False)
    print("\nDry run successful!")
    print("Rating shape:", preds[0].shape)
    print("Logits shape:", preds[1].shape)

In [ ]:
# 1. Prepare full datasets with the VRAM-safe batch size
print("Preparing full datasets for Transformer training...")
train_dataset = create_tf_dataset(
    train_idx, tokenizer, max_len=128, batch_size=32, is_training=True
)
val_dataset = create_tf_dataset(
    val_idx, tokenizer, max_len=128, batch_size=32, is_training=False
)

# 2. Set up Callbacks
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=3, restore_best_weights=True
)

# 3. Train the State-of-the-Art Architecture
print("\nStarting Transformer Decoder Training...")
history = transformer_model.fit(
    train_dataset, validation_data=val_dataset, epochs=15, callbacks=[early_stopping]
)

In [ ]:
transformer_model.save_weights("actual_hybrid_transformer2_weights.h5")

In [ ]:
# 1. Re-split the evaluation datasets from the FULLY ENCODED final_df
item_counts = final_df["parent_asin"].value_counts()
cold_items = item_counts[item_counts < 5].index.tolist()
head_items = item_counts[item_counts >= 5].index.tolist()

final_df_cold = final_df[final_df["parent_asin"].isin(cold_items)].copy()
final_df_head = final_df[final_df["parent_asin"].isin(head_items)].copy()

# 2. Convert the updated DataFrames into TF Datasets
print("Preparing evaluation datasets...")
cold_dataset = create_tf_dataset(
    final_df_cold, tokenizer, max_len=128, batch_size=16, is_training=False
)
head_dataset = create_tf_dataset(
    final_df_head, tokenizer, max_len=128, batch_size=16, is_training=False
)

# 3. Evaluate Rating Accuracy (MAE)
print("\n--- RATING PREDICTION EVALUATION ---")
print("Evaluating on HEAD items (5+ ratings)...")
head_metrics = transformer_model.evaluate(head_dataset)

print("\nEvaluating on COLD items (<5 ratings)...")
cold_metrics = transformer_model.evaluate(cold_dataset)

# 4. Evaluate Justification Quality (BLEU)
print("\n--- TEXT GENERATION (NLG) EVALUATION ---")
print("Calculating BLEU score for HEAD items...")
head_bleu = evaluate_nlg_performance(transformer_model, head_dataset, tokenizer)

print("\nCalculating BLEU score for COLD items...")
cold_bleu = evaluate_nlg_performance(transformer_model, cold_dataset, tokenizer)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# 1. Save the best weights
transformer_model.save_weights("ljmu_transformer_best_weights.h5")
print("Model weights secured!")

# 2. Plot and save the loss curves for your thesis document
history_df = pd.DataFrame(history.history)
history_df[["loss", "val_loss"]].plot(title="Total Loss Convergence")
plt.savefig("transformer_total_loss.png")

history_df[["rating_output_mae", "val_rating_output_mae"]].plot(
    title="Rating MAE Convergence"
)
plt.savefig("transformer_mae_convergence.png")

In [ ]:
import numpy as np


def sample_with_temperature(logits, temperature=0.7):
    # Scale logits by temperature (higher = more random, lower = more greedy)
    logits = logits / temperature
    probabilities = tf.nn.softmax(logits).numpy()[0]

    # Randomly sample from the probability distribution instead of pure argmax
    chosen_index = np.random.choice(len(probabilities), p=probabilities)
    return chosen_index


# Inside your generation loop, replace `np.argmax(preds)` with:
# next_token = sample_with_temperature(preds[0, -1, :], temperature=0.8)

In [ ]:
!pip install torch transformers peft datasets accelerate

In [6]:
import pandas as pd
from datasets import Dataset

# Assuming final_df is loaded and contains 'user_id', 'parent_asin', 'brand', 'category', 'name', 'rating', 'reviews_text'


def format_t5_data(row):
    # The Prompt: Injecting the KG context
    input_text = (
        f"predict rating and explain - "
        f"user: {row['user_id']} "
        f"item: {row['parent_asin']} "
        f"brand: {row['brand']} "
        f"category: {row['category']} "
        f"title: {row['name']}"
    )
    # The Target: Forcing sentiment alignment
    target_text = f"Rating: {row['rating']}. Explanation: {row['reviews_text']}"
    return {"input_text": input_text, "target_text": target_text}


print("Formatting prompt dataset...")
formatted_df = final_df.apply(format_t5_data, axis=1, result_type="expand")

# Maintain your 80/20 split logic
split_idx = int(len(formatted_df) * 0.8)
train_df = formatted_df.iloc[:split_idx]
val_df = formatted_df.iloc[split_idx:]

train_dataset = Dataset.from_pandas(train_df)
val_dataset = Dataset.from_pandas(val_df)
print(
    f"Formatted {len(train_dataset)} train and {len(val_dataset)} validation samples."
)

/home/deven/miniconda3/envs/anchit/lib/python3.12/site-packages/requests/__init__.py:92: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


Formatting prompt dataset...
Formatted 80000 train and 20000 validation samples.


In [7]:
from transformers import T5Tokenizer

tokenizer = T5Tokenizer.from_pretrained("t5-base")


def tokenize_function(examples):
    # Tokenize the input prompt
    inputs = tokenizer(
        examples["input_text"], max_length=128, truncation=True, padding="max_length"
    )

    # Tokenize the target output
    targets = tokenizer(
        examples["target_text"], max_length=128, truncation=True, padding="max_length"
    )

    # Replace padding token id with -100 for loss masking
    labels = targets["input_ids"]
    labels = [
        [-100 if token == tokenizer.pad_token_id else token for token in label]
        for label in labels
    ]

    inputs["labels"] = labels
    return inputs


print("Tokenizing datasets...")
tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_val = val_dataset.map(tokenize_function, batched=True)

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


Tokenizing datasets...


/home/deven/miniconda3/envs/anchit/lib/python3.12/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


Map:   0%|          | 0/80000 [00:00<?, ? examples/s]

Map:   0%|          | 0/20000 [00:00<?, ? examples/s]

In [ ]:
# 1. Reload the model to wipe the corrupted NaN weights
print("Reloading T5-base to clear NaN weights...")
model = T5ForConditionalGeneration.from_pretrained("t5-base").to("cuda")

peft_model = get_peft_model(model, lora_config)
peft_model.config.use_cache = False

# 2. Swap FP16 for BF16 in your arguments
training_args = Seq2SeqTrainingArguments(
    output_dir="./t5_explainability_results",
    eval_strategy="epoch",
    learning_rate=5e-4,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    weight_decay=0.01,
    save_total_limit=2,
    num_train_epochs=5,
    fp16=False,  # <-- Turn this OFF
    bf16=True,  # <-- Turn this ON (The T5 lifesaver)
    predict_with_generate=True,
    logging_steps=100,
    report_to="none",
)

# 3. Re-initialize and train
trainer = Seq2SeqTrainer(
    model=peft_model,
    args=training_args,
    train_dataset=clean_train_dataset,
    eval_dataset=clean_val_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator,
)

print("Starting T5 LoRA Training (BF16 Safe Mode)...")
trainer.train()

Reloading T5-base to clear NaN weights...
Starting T5 LoRA Training (BF16 Safe Mode)...


Epoch,Training Loss,Validation Loss
1,2.921900,2.760549
2,2.912500,2.734803
3,2.884800,2.724251
